---

# 스프린트미션16 4팀_김명환

## 1. 기본 라이브러리 / 함수
### 1.1. 라이브러리

In [19]:
!pip install --index-url https://test.pypi.org/simple/ helper-plot-hangul
!pip install --index-url https://test.pypi.org/simple/ helper-utils

Looking in indexes: https://test.pypi.org/simple/
Looking in indexes: https://test.pypi.org/simple/


In [15]:
# import importlib
# from helper_plot_hangul import helper_plot_hangul
# importlib.reload(helper_plot_hangul)

# import helper_utils.helper_logger as helper_logger
# importlib.reload(helper_logger)

# import helper_utils.helper_utils_colab as helper_utils_colab
# importlib.reload(helper_utils_colab)

from helper_plot_hangul import *
from helper_utils.helper_logger import *
from helper_utils.helper_utils_colab import *
from helper_utils.helper_utils_print import *
from helper_utils.helper_pandas import *

In [3]:
# 기본 라이브러리

# --- Scikit-learn: 데이터 전처리, 모델, 평가 ---
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import (
    fetch_california_housing, load_iris, make_moons, make_circles,
    load_breast_cancer, load_wine
)
from sklearn import datasets
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.metrics import average_precision_score

# --- 기타 라이브러리 ---
from PIL import Image
from PIL import ImageFilter
from PIL import ImageDraw
import albumentations as A
import IPython.display
#from tqdm import tqdm
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# --- PyTorch: 딥러닝 관련 ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as TF
from torch.nn import CrossEntropyLoss
from collections import OrderedDict

# --- 기타 ---
import re
import os
import sys
import copy
import json
import math
import random
import yaml
import shutil
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from datetime import datetime
from datetime import timezone, timedelta
import pytz
__kst = pytz.timezone('Asia/Seoul')

# GPU 설정
__device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
__device_cpu = torch.device('cpu')

  # 재현 가능한 결과를 위해
np.random.seed(42)
torch.manual_seed(42)
if __device == 'cuda':
    torch.cuda.manual_seed_all(42)

print(f"라이브러리 로드 완료 사용장치:{__device}")

라이브러리 로드 완료 사용장치:cpu


### > 설정 < 플레그

In [4]:
DEBUG_ON = False if IS_COLAB else True
DEBUG_ON = False
TRAIN_ON = False
logger.info(f"IS_COLAB={IS_COLAB}")
logger.info(f"DEBUG_ON={DEBUG_ON}")


2025-12-06 20:14:17 I [helper_utils_print:4] - IS_COLAB=False
2025-12-06 20:14:17 I [helper_utils_print:5] - DEBUG_ON=False


## 2. 데이터 로드

- list.txt 파싱

In [5]:
root_cache_path = my_cache()
root_my_driver = my_driver()

logger.debug(f"root_cache_path: {root_cache_path}")
logger.debug(f"root_my_driver: {root_my_driver}")

2025-12-06 20:14:17 D [helper_utils_print:4] - root_cache_path: d:\temp\cache_local
2025-12-06 20:14:17 D [helper_utils_print:5] - root_my_driver: D:\GoogleDrive


### Yolo DataSet

In [53]:
import os, sys
import importlib
sys.path.insert(0, os.getcwd())

# 기존 모듈 완전 제거
if 'yolo_eval' in sys.modules:
    del sys.modules['yolo_eval']
    
# 하위 모듈도 제거
for key in list(sys.modules.keys()):
    if key.startswith('yolo_eval.'):
        del sys.modules[key]

# 새로 임포트
from yolo_eval import *

logger.info("YOLOEvaluator 클래스 로드 완료")

2025-12-06 21:32:47 I [helper_logger:58] - EvaluationMetrics 클래스 로드 완료
2025-12-06 21:32:47 I [helper_logger:24] - PredictionResult 클래스 로드 완료
2025-12-06 21:32:47 I [helper_logger:302] - YOLOEvaluator 클래스 로드 완료
2025-12-06 21:32:47 I [helper_logger:212] - YOLOEvaluationPipeline 클래스 로드 완료
2025-12-06 21:32:47 I [helper_pandas:17] - YOLOEvaluator 클래스 로드 완료


In [54]:
yolo_dataset_path = my_cache_path("yolo", "the-oxfordiiit-pet-dataset")
logger.info(f"yolo_dataset_path: {yolo_dataset_path}")

2025-12-06 21:32:50 D [helper_utils_colab:515] - my_cache_path base: D:\temp\cache_local
2025-12-06 21:32:50 D [helper_utils_colab:582] - my_cache_path result (before create/validate): D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset
2025-12-06 21:32:50 D [helper_utils_colab:588] - Directory created: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset
2025-12-06 21:32:50 D [helper_utils_colab:598] - Path validation passed: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset
2025-12-06 21:32:50 I [helper_pandas:2] - yolo_dataset_path: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset


In [55]:

yaml_path, train_df, valid_df, test_df, validation_results = oxfordiit_pet_to_yolo(max_samples_per_split=(30, 20, 10),
                                                                                   label_mode="species",
                                                                                   output_dir=Path(yolo_dataset_path))


2025-12-06 21:32:52,585 - yolo_eval.oxfordiiit_pet_dataset - INFO - Kaggle 데이터셋 다운로드 시작
2025-12-06 21:32:53,154 - yolo_eval.oxfordiiit_pet_dataset - INFO - 다운로드 완료: C:\Users\sw1\.cache\kagglehub\datasets\devdgohil\the-oxfordiiit-pet-dataset\versions\2
2025-12-06 21:32:53,155 - yolo_eval.oxfordiiit_pet_dataset - INFO - YOLO 데이터셋 변환 시작
2025-12-06 21:32:53,160 - yolo_eval.oxfordiiit_pet_dataset - INFO - 샘플 개수 제한 적용: 각 분할당 최대 (30, 20, 10)개
2025-12-06 21:32:53,163 - yolo_eval.oxfordiiit_pet_dataset - INFO - Train: 30, Val: 20, Test: 10
Processing test: 100%|██████████| 10/10 [00:00<00:00, 1795.74it/s]
2025-12-06 21:32:53,204 - yolo_eval.oxfordiiit_pet_dataset - INFO - YOLO 데이터셋 생성 완료: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\data.yaml
2025-12-06 21:32:53,205 - yolo_eval.oxfordiiit_pet_dataset - INFO - YOLO 데이터셋을 DataFrame으로 로드
2025-12-06 21:32:53,217 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\labels\train\Abyssinia

In [56]:
logger.info(f"yaml_path: {yaml_path}")
print(f"train_df")
train_df.head(2)
print("-----")
print("valid_df")
valid_df.head(2)
print("-----")
print("test_df")
test_df.head(2)

2025-12-06 21:33:09 I [helper_pandas:1] - yaml_path: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\data.yaml
train_df
                                                                           image_path    image_name class_id class_name x_center y_center  width height split
 0  D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\images\train\Abyssinian_1.jpg  Abyssinian_1        0        cat   0.6317   0.2875 0.1533  0.215 train
 1 D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\images\train\Abyssinian_10.jpg Abyssinian_10        0        cat     0.48    0.396  0.576  0.372 train
-----
valid_df
                                                                          image_path     image_name class_id class_name x_center y_center width height split
 0 D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\images\val\leonberger_183.jpg leonberger_183        1        dog    0.599    0.372 0.294  0.328   val
 1 D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\images\val\leonber

In [62]:
yolov8m_path = my_driver_path('modeling', 'model', 'modeling16', 'yolov8m_20251205_2005')
#yolov8m_model_path = my_driver_path(yolov8m_path, 'weights', 'best.pt', create=False)
yolov8m_model_path = my_driver_path(yolov8m_path, 'out', 'mission_16_yolo.pth', create=False)
logger.info(f"yolov8m_model_path: {yolov8m_model_path}")

2025-12-06 21:36:27 D [helper_utils_colab:329] - my_driver_path base: D:\GoogleDrive
2025-12-06 21:36:27 D [helper_utils_colab:378] - my_driver_path result (before create/validate): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005
2025-12-06 21:36:27 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005
2025-12-06 21:36:27 D [helper_utils_colab:394] - Path validation passed: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005
2025-12-06 21:36:27 D [helper_utils_colab:396] - my_driver_path final: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005
2025-12-06 21:36:27 D [helper_utils_colab:329] - my_driver_path base: D:\GoogleDrive
2025-12-06 21:36:27 D [helper_utils_colab:347] - First subpath is absolute, using as start: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005
2025-12-06 21:36:27 D [helper_utils_colab:378] - my_driver_path result (before create/validate): D:\GoogleDrive\mo

## 3. 평가

In [63]:
# 파이프라인 생성
logger.info(f"yaml_path: {yaml_path}")
logger.info(f"yolov8m_path: {yolov8m_path}")
logger.info(f"yolov8m_best_path: {yolov8m_model_path}")

pipeline = YOLOEvaluationPipeline(
    yaml_path=yaml_path,
    device=str(__device)
)

# 평가할 모델 추가
pipeline.add_model(
    model_path=yolov8m_model_path,
    model_name="YOLOv8m_baseline",
    verbose=False
)



2025-12-06 21:36:33 I [helper_pandas:2] - yaml_path: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\data.yaml
2025-12-06 21:36:33 I [helper_pandas:3] - yolov8m_path: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005
2025-12-06 21:36:33 I [helper_pandas:4] - yolov8m_best_path: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out\mission_16_yolo.pth
2025-12-06 21:36:33 I [helper_logger:55] - 모델 추가: YOLOv8m_baseline
2025-12-06 21:36:33 I [helper_logger:61] - 모델 로딩 중: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out\mission_16_yolo.pth
WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
2025-12-06 21:36:33 I [helper_logger:63] - 모델 로드 완료: YOLOv8m_baseline
2025-12-06 21:36:33 I [helper_logger:74] - 데이터셋 경로: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset
2025-12-06 21:36:33 I [helper_logger:75] - 테스트 이미지: D:\temp\cache_

In [65]:
out_qint8_dir_path = my_driver_path(yolov8m_path, 'out_qint8', create=True)
out_result_dir_path = my_driver_path(yolov8m_path, 'out', 'result', create=True)
logger.info(f"output_qint8_dir_path: {out_qint8_dir_path}")
logger.info(f"output_result_dir_path: {out_result_dir_path}")

2025-12-06 21:37:51 D [helper_utils_colab:329] - my_driver_path base: D:\GoogleDrive
2025-12-06 21:37:51 D [helper_utils_colab:347] - First subpath is absolute, using as start: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005
2025-12-06 21:37:51 D [helper_utils_colab:378] - my_driver_path result (before create/validate): D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out_qint8
2025-12-06 21:37:51 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out_qint8
2025-12-06 21:37:51 D [helper_utils_colab:394] - Path validation passed: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out_qint8
2025-12-06 21:37:51 D [helper_utils_colab:396] - my_driver_path final: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out_qint8
2025-12-06 21:37:51 D [helper_utils_colab:329] - my_driver_path base: D:\GoogleDrive
2025-12-06 21:37:51 D [helper_utils_colab:347] - First subpath is absolute

In [ ]:
raise Exception("STOP")

In [66]:

# 전체 평가 실행
results = pipeline.run_full_evaluation(
    val_kwargs={
        'project': out_result_dir_path,  # custom_results/ 디렉터리에
        # 'name': 'baseline_test',      # baseline_test/ 하위 폴더로 저장
        'split': 'test',
        'imgsz': 640,
        'batch': 16,
        'conf': 0.25,
        'iou': 0.75
    },
    # pred_max_images=100,
    pred_conf=0.25
)


모델 검증 시작

[YOLOv8m_baseline] 검증 중...
2025-12-06 21:37:55 I [helper_logger:124] - 모델 검증 시작: split=test, imgsz=640, batch=16
Ultralytics 8.3.235  Python-3.10.18 torch-2.8.0+cpu CPU (12th Gen Intel Core(TM) i7-1260P)


AssertionError: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\out\mission_16_yolo.pth acceptable suffix is {'.pt'}, not .pth

In [34]:
print_dic_tree(results)

2025-12-06 20:38:56 I [helper_utils_print:443] - ├─ YOLOv8m_baseline [EvaluationMetrics]: EvaluationMetrics(map50=0.995, map50_95=0.9404142857142856, precision=1.0, recall=1.0, inference_tim...


In [35]:

# 결과 출력
pipeline.print_summary()

평가 결과: YOLOv8m_baseline
mAP50: 0.9950
mAP50-95: 0.9404
정밀도(Precision): 1.0000
재현율(Recall): 1.0000
추론 시간(평균): 635.12ms
--------------------------------------------------------------------------------
테스트된 총 이미지 수: 10
GT 박스가 있는 이미지 수: 10
예측이 있는 이미지 수: 10
이미지당 평균 GT 박스 수: 1.00
이미지당 평균 예측 수: 1.00


In [38]:
# 비교 DataFrame 생성
comparison_df = pipeline.get_comparison_dataframe()
logger.info("모델 비교 결과:")
display(comparison_df)

2025-12-06 20:39:39 I [helper_pandas:3] - 모델 비교 결과:


,model_name,mAP50,mAP50-95,precision,recall,inference_time_ms,total_images,images_with_gt,images_with_pred,avg_gt_boxes,avg_pred_boxes
0,YOLOv8m_baseline,0.995,0.940414,1.0,1.0,635.11641,10,10,10,1.0,1.0


In [41]:
# 개별 이미지 예측 결과 확인
evaluator = pipeline.evaluators["YOLOv8m_baseline"]
results_df = evaluator.get_results_dataframe()
logger.info("개별 이미지 예측 결과 (상위 5개):")
results_df.head(5)

2025-12-06 20:40:26 I [helper_pandas:4] - 개별 이미지 예측 결과 (상위 5개):
      image_name gt_count pred_count inference_time_ms
 0 shiba_inu_129        1          1          309.8474
 1  shiba_inu_13        1          1          449.6317
 2 shiba_inu_130        1          1           779.201
 3 shiba_inu_131        1          1          745.3425
 4 shiba_inu_132        1          1          652.9796


### 3.4. 양자화 모델 평가 예제 (준비)

In [ ]:
# 양자화 모델 평가 예제 (양자화 모델이 있을 때 사용)
"""
# 파이프라인에 여러 모델 추가
pipeline_multi = YOLOEvaluationPipeline(
    yaml_path=yaml_path,
    device=str(__device)
)

# 원본 모델
pipeline_multi.add_model(
    model_path=yolov8m_best_path,
    model_name="YOLOv8m_baseline"
)

# 양자화 모델 (예시)
# quantized_model_path = my_driver_path('modeling', 'model', 'modeling16', 
#                                       'yolov8m_quantized', 'weights', 'best.pt')
# pipeline_multi.add_model(
#     model_path=quantized_model_path,
#     model_name="YOLOv8m_int8"
# )

# 전체 평가 실행
# results_multi = pipeline_multi.run_full_evaluation(
#     val_kwargs={'split': 'test', 'imgsz': 640, 'batch': 16},
#     pred_max_images=10
# )

# 모델 비교
# comparison_df_multi = pipeline_multi.get_comparison_dataframe()
# print("\n모델 성능 비교:")
# display(comparison_df_multi)

# 결과 저장
# output_path = my_driver_path('modeling', 'evaluation', 'model_comparison')
# pipeline_multi.save_results(output_path)
"""

print("양자화 모델 평가 준비 완료")

In [ ]:
# (레거시) 기존 방식의 단순 테스트 코드
# 새로운 클래스 기반 평가는 위의 파이프라인을 사용하세요

"""
# 간단한 단일 모델 평가 (레거시)
evaluator_simple = YOLOEvaluator(
    model_path=yolov8m_best_path,
    yaml_path=yaml_path,
    model_name="YOLOv8m_simple_test",
    device=str(__device)
)

# 검증만 실행
metrics_simple = evaluator_simple.validate(
    split='test',
    imgsz=640,
    batch=16,
    conf=0.25,
    iou=0.75
)

metrics_simple.print_summary()
"""

print("레거시 테스트 코드 (주석 처리됨)")